# TP 3 — Une table Iceberg sur stockage objet### Module 2 — Stockage distribué · Big Data M2 / Ingénieur**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin de ce TP1. Créer une table Iceberg sur MinIO, avec un **partitionnement caché**.2. Inspecter les métadonnées : instantanés, manifestes, fichiers.3. Faire évoluer un schéma — y compris **renommer une colonne** — sans réécriture.4. Mettre à jour et supprimer des lignes, et observer ce qui se passe réellement.5. Voyager dans le temps, puis **expirer** les instantanés.6. Expliquer pourquoi une suppression ne suffit pas au droit à l'effacement.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Créer la table et l'alimenter | 3 || 2 | Anatomie des métadonnées | 4 || 3 | Évolution de schéma | 4 || 4 | Mise à jour, suppression, instantanés | 4 || 5 | Voyage dans le temps et maintenance | 4 || 6 | Synthèse RGPD | 1 |

---# Exercice 1 — Créer la table  *(3 points)*

In [ ]:
# 1.1 — Session Spark avec le catalogue Iceberg sur MinIOfrom pyspark.sql import SparkSessionimport timeUTILISATEUR = "etudiant"        # <-- votre nom de famillespark = (SparkSession.builder    .appName("TP3 - Iceberg")    .master("local[*]")    # Extensions SQL d'Iceberg : CALL, MERGE INTO, ALTER ... RENAME COLUMN    .config("spark.sql.extensions",            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")    .config("spark.sql.catalog.lakehouse.type", "hadoop")    .config("spark.sql.catalog.lakehouse.warehouse", f"s3a://lakehouse/{UTILISATEUR}")    # Accès MinIO    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")    .config("spark.hadoop.fs.s3a.path.style.access", "true")    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")    .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version)

In [ ]:
# 1.2 — Créer la base et la table, partitionnée par jour SANS colonne de partitionspark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.finance")spark.sql("DROP TABLE IF EXISTS lakehouse.finance.transactions")spark.sql("""CREATE TABLE lakehouse.finance.transactions (    id_transaction   STRING,    horodatage       TIMESTAMP,    id_compte        STRING,    id_marchand      STRING,    montant          DOUBLE,    devise           STRING,    pays_transaction STRING,    est_fraude       BOOLEAN) USING icebergPARTITIONED BY (days(horodatage))""")spark.sql("DESCRIBE EXTENDED lakehouse.finance.transactions").show(50, truncate=False)

In [ ]:
# 1.3 — Alimenter la table depuis le jeu de données du TP2!python /home/tinku/cours/99-Infra/scripts/generate_datasets.py \        --filiere if --sortie /home/tinku/work/data --evenements 1000000from pyspark.sql.functions import to_timestamp, colsrc = (spark.read.json("file:///home/tinku/work/data/if_transactions.jsonl")       .select("id_transaction",               to_timestamp("horodatage").alias("horodatage"),               "id_compte", "id_marchand",               col("montant").cast("double").alias("montant"),               "devise", "pays_transaction", "est_fraude"))src.writeTo("lakehouse.finance.transactions").append()print("lignes :", spark.table("lakehouse.finance.transactions").count())

### Q1 *(3 pts)* — Regardez la sortie de `DESCRIBE EXTENDED`.- **a.** La table contient-elle une colonne `jour` ou `date` ? Comment le partitionnement  est-il alors exprimé ?- **b.** Où les fichiers ont-ils été écrits ? Vérifiez dans la console MinIO  (http://localhost:9001, `minioadmin`/`minioadmin`) et décrivez l'arborescence.- **c.** Quelle différence essentielle avec un partitionnement Hive classique ?*(rédigez ici)*

---# Exercice 2 — Anatomie des métadonnées  *(4 points)*Iceberg expose ses métadonnées comme des tables interrogeables en SQL. C'est lemeilleur moyen de comprendre son fonctionnement.

In [ ]:
# 2.1 — Les instantanésspark.sql("SELECT * FROM lakehouse.finance.transactions.snapshots").show(truncate=False)

In [ ]:
# 2.2 — Les fichiers de données, avec leurs statistiques(spark.sql("""   SELECT file_path, record_count, file_size_in_bytes, partition   FROM lakehouse.finance.transactions.files   ORDER BY file_size_in_bytes DESC""").show(10, truncate=False))print("nombre de fichiers :",      spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0])

In [ ]:
# 2.3 — L'historique et les manifestesspark.sql("SELECT * FROM lakehouse.finance.transactions.history").show(truncate=False)spark.sql("SELECT path, added_data_files_count FROM lakehouse.finance.transactions.manifests") \     .show(truncate=False)

### Q2 *(4 pts)* —- **a.** Combien d'instantanés existe-t-il après un seul `append` ? Que contient la  colonne `summary` ?- **b.** Combien de fichiers de données ? Comparez au nombre de partitions (jours  distincts). Que remarquez-vous sur leur taille ?- **c.** Dessinez la chaîne qui relie le catalogue au premier fichier Parquet, en  nommant chaque niveau.*(rédigez ici)*

---# Exercice 3 — Faire évoluer le schéma  *(4 points)*

In [ ]:
# 3.1 — Compter les fichiers AVANT, pour pouvoir compareravant = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0]print("fichiers avant :", avant)

In [ ]:
# 3.2 — Trois évolutions de schéma, chronométréesfor sql in [    "ALTER TABLE lakehouse.finance.transactions ADD COLUMN canal STRING",    "ALTER TABLE lakehouse.finance.transactions RENAME COLUMN montant TO montant_eur",    "ALTER TABLE lakehouse.finance.transactions ALTER COLUMN id_compte COMMENT 'IBAN masqué'",]:    t0 = time.time()    spark.sql(sql)    print(f"{time.time()-t0:5.2f} s  |  {sql[:70]}")apres = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0]print("fichiers après :", apres)

In [ ]:
# 3.3 — Vérifier que les anciennes données sont toujours lisiblesspark.sql("""  SELECT id_transaction, montant_eur, canal  FROM lakehouse.finance.transactions  LIMIT 5""").show(truncate=False)

### Q3 *(4 pts)* —- **a.** Combien de temps a pris le renommage ? Combien de fichiers ont été réécrits ?- **b.** Les fichiers Parquet sur disque contiennent toujours une colonne nommée  `montant`. Comment Spark parvient-il à la lire sous le nom `montant_eur` ?- **c.** Que vaut la colonne `canal` sur les lignes écrites **avant** son ajout ? Pourquoi ?- **d.** Tentez `ALTER TABLE ... ALTER COLUMN montant_eur TYPE INT`. Que se passe-t-il,  et pourquoi ?*(rédigez ici)*

---# Exercice 4 — Mise à jour, suppression, instantanés  *(4 points)*

In [ ]:
# 4.1 — État de départdef etat(titre):    n_lignes = spark.table("lakehouse.finance.transactions").count()    n_fich   = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0]    n_snap   = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.snapshots").first()[0]    print(f"{titre:28s} lignes={n_lignes:>9,d}  fichiers={n_fich:>5d}  instantanés={n_snap:>3d}")etat("départ")

In [ ]:
# 4.2 — Un UPDATE ciblé, puis un DELETEspark.sql("""  UPDATE lakehouse.finance.transactions  SET devise = 'EUR'  WHERE devise = 'XOF'""")etat("après UPDATE")spark.sql("""  DELETE FROM lakehouse.finance.transactions  WHERE montant_eur IS NULL""")etat("après DELETE")

In [ ]:
# 4.3 — Ce que raconte l'historiquespark.sql("""  SELECT snapshot_id, operation,         summary['added-data-files']   AS ajoutes,         summary['deleted-data-files'] AS supprimes,         summary['total-records']      AS total  FROM lakehouse.finance.transactions.snapshots  ORDER BY committed_at""").show(truncate=False)

### Q4 *(4 pts)* —- **a.** Le nombre de fichiers a-t-il augmenté ou diminué après l'`UPDATE` ? Pourquoi ?- **b.** Le `DELETE` a-t-il réellement effacé des octets sur MinIO ? Vérifiez dans la  console, et expliquez.- **c.** Quelle stratégie Iceberg applique-t-il ici — Copy-on-Write ou Merge-on-Read ?  Sur quel indice le voyez-vous ?*(rédigez ici)*

---# Exercice 5 — Voyage dans le temps et maintenance  *(4 points)*

In [ ]:
# 5.1 — Interroger un état passésnaps = spark.sql("""  SELECT snapshot_id, committed_at  FROM lakehouse.finance.transactions.snapshots ORDER BY committed_at""").collect()premier = snaps[0]["snapshot_id"]print("premier instantané :", premier)n_now = spark.table("lakehouse.finance.transactions").count()n_old = (spark.read.option("snapshot-id", premier)         .format("iceberg").load("lakehouse.finance.transactions").count())print(f"aujourd'hui : {n_now:,}   |   au premier instantané : {n_old:,}")

In [ ]:
# 5.2 — Compacter les petits fichiersavant = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0]spark.sql("""  CALL lakehouse.system.rewrite_data_files(      table => 'finance.transactions',      options => map('target-file-size-bytes', '134217728')  )""").show(truncate=False)apres = spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.files").first()[0]print(f"fichiers : {avant} -> {apres}")

In [ ]:
# 5.3 — Expirer les anciens instantanés, puis nettoyer les orphelinsfrom datetime import datetime, timedelta, timezonelimite = (datetime.now(timezone.utc) + timedelta(seconds=5)).strftime("%Y-%m-%d %H:%M:%S")spark.sql(f"""  CALL lakehouse.system.expire_snapshots(      table => 'finance.transactions',      older_than => TIMESTAMP '{limite}',      retain_last => 1  )""").show(truncate=False)spark.sql("SELECT count(*) FROM lakehouse.finance.transactions.snapshots").show()

In [ ]:
# 5.4 — Le voyage dans le temps fonctionne-t-il encore ?try:    n = (spark.read.option("snapshot-id", premier)         .format("iceberg").load("lakehouse.finance.transactions").count())    print("lecture réussie :", n)except Exception as e:    print("ÉCHEC :", type(e).__name__)    print(str(e)[:300])

### Q5 *(4 pts)* —- **a.** La compaction a-t-elle changé le nombre de lignes ? Le nombre de fichiers ?  A-t-elle créé un nouvel instantané ?- **b.** Après `expire_snapshots`, que se passe-t-il à la cellule 5.4 ? Pourquoi ?- **c.** Quelle est la conséquence de cette expérience pour une politique de sauvegarde ?*(rédigez ici)*

---# Exercice 6 — Synthèse RGPD  *(1 point)*Un client exerce son droit à l'effacement. Vous exécutez :```sqlDELETE FROM lakehouse.finance.transactions WHERE id_compte = 'C0001234';```### Q6 — Cette opération suffit-elle à satisfaire l'obligation légale ?Si non, indiquez la séquence complète des opérations nécessaires, dans l'ordre.

*(rédigez ici)*

In [ ]:
spark.stop()print("Session fermée.")

---## Avant de rendre- [ ] Toutes les cellules exécutées dans l'ordre.- [ ] Les six questions rédigées.- [ ] Une capture de la console MinIO montrant l'arborescence `data/` et `metadata/`.- [ ] Notebook exporté en HTML.